[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_cpgptgrimage3.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_cpgptgrimage3.ipynb)

# The best DNAm mortality predictor: CpGPTGrimAge3

## Table of Contents

0. <a href="#0-read-quick-setup-tutorial">Read Quick Setup Tutorial</a>
1. <a href="#1-setup-environment">Setup Environment</a>
2. <a href="#2-load-data">Load Data</a>
3. <a href="#3-load-model-and-dependencies">Load Model and Dependencies</a>
4. <a href="#4-prepare-data-objects">Prepare Data Objects</a>
5. <a href="#5-compute-protein-proxies">Compute Protein Proxies</a>
6. <a href="#6-calculate-cpgptgrimage3">Calculate CpGPTGrimAge3</a>

## 0. Read Quick Setup Tutorial

Before, going through this tutorial, please familiarize yourself with the [quick setup tutorial](https://github.com/lcamillo/CpGPT/blob/main/tutorials/quick_setup.ipynb).

## 1. Setup Environment

CpGPT needs to be installed. The easiest is to use the following:

In [1]:
!pip install cpgpt>=0.0.12 --quiet

zsh:1: 0.0.12 not found


Please check out more instructions in the [offical CpGPT repo](https://github.com/lucascamillomd/CpGPT).

We'll import the necessary Python packages and set up our environment for CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements.

CpGPT model files and DNA-sequence dependencies are hosted on Hugging Face.
The next Python cell downloads any missing files into the standard Hugging Face cache and reuses them on later runs; no command-line download is needed.

In [2]:
from cpgpt import download_cpgpt

resources = download_cpgpt(model="proteins", species="human")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
DEPENDENCIES_DIR = "../dependencies"
DATA_DIR = "../data"
PROCESSED_DIR = "../data/tutorials/processed/predict_mortality"

MODEL_NAME = "proteins" # this is the name of the model checkpoint required for CpGPTGrimAge3

BETAS_PATH = "../data/cpgcorpus/raw/GSE237561/GPL13534/betas/QCDPB.arrow"
FILTERED_BETAS_PATH = "../data/cpgcorpus/raw/GSE237561/GPL13534/betas/QCDPB_filtered.arrow"
METADATA_PATH = "../data/cpgcorpus/raw/GSE237561/GPL13534/metadata/metadata.arrow"

# The maximum context length to give to the model
MAX_INPUT_LENGTH = 10_000 # you might wanna go higher hardware permitting

LLM_DEPENDENCIES_DIR = str(resources.dependencies_path)
MODEL_CHECKPOINT_PATH = str(resources.checkpoint_path)
MODEL_CONFIG_PATH = str(resources.config_path)
MODEL_VOCAB_PATH = str(resources.vocab_path) if resources.vocab_path is not None else None

> **⚠️ Warning**
> 
> It is recommended to have a GPU for inference as CPU might be slow.
> 
> Reconstructing the methylome for a few hundred samples might take up to one hour on a CPU. ⌛
>
> This might be a great exercise in testing your patience.

### 1.2 Import packages


In [4]:
# Standard library imports
import warnings
import os
import json

warnings.simplefilter(action="ignore", category=FutureWarning)

# Plotting imports
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyaging as pya
import seaborn as sns
from tqdm.rich import tqdm

# Lightning imports
from lightning.pytorch import seed_everything

# cpgpt-specific imports
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver
from cpgpt.data.cpgpt_datamodule import CpGPTDataModule
from cpgpt.trainer.cpgpt_trainer import CpGPTTrainer
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from cpgpt.model.cpgpt_module import m_to_beta

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)
try:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except:
    pass

Seed set to 42


## 2. Load Data

If you have your own data, please feel free to skip the following step but make sure it is saved in a .arrow format. Here, as an example target dataset, we'll use GSE237561, which contains methylation profiling data from 126 peripheral whole blood samples collected from 26 individuals across two independent cohorts. These samples were collected at three timepoints: prior to clozapine initiation, 4-12 weeks after initiation, and 6 months after initiation.

In [5]:
# First let's declare the inferencer
inferencer = CpGPTInferencer(dependencies_dir=DEPENDENCIES_DIR, data_dir=DATA_DIR)

inferencer.download_cpgcorpus_dataset("GSE237561")

cpgpt -CpGPTInferencer: Initializing class CpGPTInferencer.
cpgpt -CpGPTInferencer: Using device: cpu.
cpgpt -CpGPTInferencer: Using dependencies directory: ../dependencies
cpgpt -CpGPTInferencer: Using data directory: ../data
cpgpt -CpGPTInferencer: There are 2089 GSE datasets available such as GSE100184, GSE100208, GSE100209, etc.
cpgpt -CpGPTInferencer: All 4 files for dataset GSE237561 already exist at ../data/cpgcorpus/raw/GSE237561. Skipping download.


In [6]:
# Load betas matrix
df = pd.read_feather(BETAS_PATH)
df.set_index("GSM_ID", inplace=True)
df = df.iloc[:8] # filtering to speed up the tutorial

df.head()

,cg00000029,cg00000108,cg00000109,cg00000165,cg00000236,cg00000289,cg00000292,cg00000321,cg00000363,cg00000622,...,rs7746156,rs798149,rs845016,rs877309,rs9292570,rs9363764,rs939290,rs951295,rs966367,rs9839873
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM7625568,0.592157,0.964411,0.899373,NaN,0.861353,NaN,0.894893,0.280911,0.386535,0.017273,...,0.974810,0.022425,0.086804,0.031910,0.535025,0.548239,0.064924,0.536217,0.091576,0.787829
GSM7625569,0.657346,0.962779,0.920897,0.170290,0.868804,NaN,0.945775,0.303094,0.409573,0.015473,...,0.977200,0.023908,0.093790,0.024449,0.524899,0.554774,0.048559,0.521857,0.081250,0.772327
GSM7625570,0.662022,0.964065,0.903984,0.180436,0.867933,NaN,0.915102,0.241706,0.392485,0.015086,...,0.979132,0.022763,0.091286,0.028332,0.518550,0.584879,0.054734,0.529968,0.101047,0.765933
GSM7625571,0.599778,0.961087,0.903260,NaN,0.845338,NaN,0.910445,0.277753,0.405914,0.016514,...,0.978393,0.019485,0.069463,0.024282,0.508195,0.569030,0.052701,0.508199,0.083252,0.787774
GSM7625572,0.556610,0.960655,0.893885,NaN,0.846172,NaN,0.916346,0.285945,0.404618,0.014193,...,0.978121,0.019678,0.541755,0.982736,0.528156,0.524965,0.965231,0.041688,0.672611,0.924847


In [7]:
# Load metadata
metadata = pd.read_feather(METADATA_PATH)
metadata.set_index("GSM_ID", inplace=True)
metadata = metadata.iloc[:8] # filtering to speed up the tutorial

metadata.head()

,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,characteristics_ch1,...,cd8t:ch1,days.on.clozapine:ch1,gran:ch1,institute:ch1,mono:ch1,nk:ch1,participant_id:ch1,Sex:ch1,smokingscore:ch1,visit:ch1
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM7625568,genomic DNA from ID 0003 for 'a' visit,GSM7625568,Public on Jul 17 2024,Jul 17 2023,Jul 17 2024,genomic,1,peripheral whole blood,Homo sapiens,participant_id: 0003,...,0.124088032132122,0,0.623747063903508,KCL,0.0577366406423333,0.0181886611163226,0003,M,0.744140048408035,a
GSM7625569,genomic DNA from ID 0003 for 'b' visit,GSM7625569,Public on Jul 17 2024,Jul 17 2023,Jul 17 2024,genomic,1,peripheral whole blood,Homo sapiens,participant_id: 0003,...,0.140642508939063,42,0.532222309707589,KCL,0.0794055065754302,0.0165444940880789,0003,M,0.778521295727892,b
GSM7625570,genomic DNA from ID 0003 for 'd' visit,GSM7625570,Public on Jul 17 2024,Jul 17 2023,Jul 17 2024,genomic,1,peripheral whole blood,Homo sapiens,participant_id: 0003,...,0.112389247455345,84,0.610861991010505,KCL,0.0694956639909667,0.00363827604122652,0003,M,0.591346224352136,d
GSM7625571,genomic DNA from ID 0003 for 'e' visit,GSM7625571,Public on Jul 17 2024,Jul 17 2023,Jul 17 2024,genomic,1,peripheral whole blood,Homo sapiens,participant_id: 0003,...,0.0927185883637476,168,0.647578326303527,KCL,0.0886719923597006,0.0163329200193279,0003,M,-0.771644717383253,e
GSM7625572,genomic DNA from ID 0005 for 'a' visit,GSM7625572,Public on Jul 17 2024,Jul 17 2023,Jul 17 2024,genomic,1,peripheral whole blood,Homo sapiens,participant_id: 0005,...,0.109718862397854,0,0.505235489278915,KCL,0.0514219148451151,0.0610566697720137,0005,M,1.32031205622569,a


## 3. Load Model and Dependencies

In order to calculate CpGPTGrimAge3, we need to calculate several DNA methylation plasma protein proxies with a finetuned model. The checkpoint is called `proteins` and it predicts 322 plasma protein levels which are normalized with mean 0 and variance 1 (μ = 0, σ² = 1).

### 3.1 Load Model

In [8]:
# Load the model configuration
config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)

# Load the model weights
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

cpgpt -CpGPTInferencer: Loaded CpGPT model config.
cpgpt -CpGPTInferencer: Instantiated CpGPT model from config.
cpgpt -CpGPTInferencer: Using device: cpu.
cpgpt -CpGPTInferencer: Loading checkpoint from: /Users/lucascamillo/.cache/huggingface/hub/models--lucascamillomd--cpgpt-models/snapshots/159217abe074b1693c1bdb8cfae309b934592742/weights/proteins.ckpt
cpgpt -CpGPTInferencer: Checkpoint loaded into the model.


### 3.2 Load Vocab

The `proteins` model was trained with a vocabulary of about 4,689 CpG sites. Ideally, the data would be filtered to only include those features (or a subset thereof).

In [9]:
# Load the vocabulary
with open(MODEL_VOCAB_PATH, "r") as f:
    vocab = json.load(f)

In [10]:
model_input_features = vocab['input']

model_input_features[:5]

['cg21830050', 'cg10381813', 'cg08067365', 'cg09864227', 'cg07213830']

In [11]:
model_output_features = vocab['output']

model_output_features[:5]

['cpgpt_tnfsf13', 'cpgpt_il33', 'cpgpt_calca', 'cpgpt_npy', 'cpgpt_hla-dra']

## 4. Prepare Data Objects

### 4.1 Declare Embedder and Prober

In order to retrieve the sample embeddings, we need to memory-map the data. This is done by using the `CpGPTDataSaver` class. We first need to define the `DNALLMEmbedder` and `IlluminaMethylationProber` classes, which contain the information about the DNA LLM Embeddings and the conversion between Illumina array probes to genomic locations, respectively.

In [12]:
embedder = DNALLMEmbedder(dependencies_dir=LLM_DEPENDENCIES_DIR)

cpgpt -DNALLMEmbedder: Initializing class DNALLMEmbedder.
cpgpt -DNALLMEmbedder: Genome files will be stored under /Users/lucascamillo/.cache/huggingface/hub/models--lucascamillomd--cpgpt-human-dependencies/snapshots/e66bbe467b3404c1a783b1361e565c5fe6c7235f/genomes.
cpgpt -DNALLMEmbedder: DNA embeddings will be stored under /Users/lucascamillo/.cache/huggingface/hub/models--lucascamillomd--cpgpt-human-dependencies/snapshots/e66bbe467b3404c1a783b1361e565c5fe6c7235f/dna_embeddings and subdirectories.
cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully


In [13]:
prober = IlluminaMethylationProber(dependencies_dir=LLM_DEPENDENCIES_DIR, embedder=embedder)

cpgpt -IlluminaMethylationProber: Initializing class IlluminaMethylationProber.
cpgpt -IlluminaMethylationProber: Illumina methylation manifest files will be stored under /Users/lucascamillo/.cache/huggingface/hub/models--lucascamillomd--cpgpt-human-dependencies/snapshots/e66bbe467b3404c1a783b1361e565c5fe6c7235f/manifests.
cpgpt -IlluminaMethylationProber: Illumina metadata dictionary loaded successfully.


### 4.2 Filter Vocab

In [14]:
common_features = list(set(model_input_features) & set(df.columns))
df_filtered = df.loc[:, common_features]
df_filtered.to_feather(FILTERED_BETAS_PATH)

df_filtered.head()

,cg03788610,cg08908247,cg21058506,cg22277972,cg11173131,cg07073964,cg16730484,cg06295856,cg27209729,cg21946374,...,cg11254439,cg10778288,cg18915856,cg11804928,cg21244086,cg11690884,cg13594542,cg07165260,cg14209920,cg13650654
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM7625568,0.947073,0.382738,0.035081,0.081597,0.281572,0.598748,0.933576,0.126092,0.657009,0.438133,...,0.052494,0.087975,0.422177,0.338496,0.436588,0.304581,0.323605,0.729406,0.815286,0.829144
GSM7625569,0.955147,0.433442,0.031778,0.081195,0.317102,0.599041,0.950659,0.154252,0.752990,0.511987,...,0.038427,0.068649,0.469453,0.334664,0.475367,0.338490,0.361028,0.708981,0.787066,0.797193
GSM7625570,0.917246,0.407013,0.029595,0.080354,0.275653,0.675376,0.928559,0.130102,0.694356,0.443145,...,0.050294,0.092176,0.413305,0.300501,0.441728,0.327890,0.369716,0.730609,0.805528,0.815616
GSM7625571,0.946284,0.353159,0.030010,0.091421,0.237429,0.557033,0.938736,0.126639,0.624799,0.415825,...,0.053471,0.074641,0.410412,0.292629,0.385209,0.249011,0.283559,0.770744,0.771077,0.792418
GSM7625572,0.951041,0.451227,0.037389,0.095143,0.342438,0.692330,0.888841,0.123183,0.708604,0.470520,...,0.054086,0.059444,0.545169,0.376101,0.428877,0.316745,0.316117,0.709906,0.799133,0.745559


### 4.3 Memory-Map Data

In [15]:
# Define datasaver
datasaver = CpGPTDataSaver(data_paths=FILTERED_BETAS_PATH, processed_dir=PROCESSED_DIR)

# Process the file
datasaver.process_files(prober, embedder)

cpgpt -CpGPTDataSaver: Initializing class CpGPTDataSaver.
cpgpt -CpGPTDataSaver: Dataset folders will be stored under ../data/tutorials/processed/predict_mortality.
cpgpt -CpGPTDataSaver: No existing dataset metrics found. Please process files.
cpgpt -CpGPTDataSaver: No existing genomic locations found. Please process files.
cpgpt -CpGPTDataSaver: Starting file processing.


Output()

cpgpt -CpGPTDataSaver: No species column found. Defaulting to homo_sapiens.


cpgpt -CpGPTDataSaver: File processing completed.


### 4.4 Declare data module

Let's define one data module to use with our model:

In [16]:
# Define datamodule
datamodule = CpGPTDataModule(
    predict_dir=PROCESSED_DIR,
    dependencies_dir=LLM_DEPENDENCIES_DIR,
    batch_size=1,
    num_workers=0,
    max_length=MAX_INPUT_LENGTH,
    dna_llm=config.data.dna_llm,
    dna_context_len=config.data.dna_context_len,
    sorting_strategy=config.data.sorting_strategy,
    pin_memory=False,
)

cpgpt -DNALLMEmbedder: Initializing class DNALLMEmbedder.
cpgpt -DNALLMEmbedder: Genome files will be stored under /Users/lucascamillo/.cache/huggingface/hub/models--lucascamillomd--cpgpt-human-dependencies/snapshots/e66bbe467b3404c1a783b1361e565c5fe6c7235f/genomes.
cpgpt -DNALLMEmbedder: DNA embeddings will be stored under /Users/lucascamillo/.cache/huggingface/hub/models--lucascamillomd--cpgpt-human-dependencies/snapshots/e66bbe467b3404c1a783b1361e565c5fe6c7235f/dna_embeddings and subdirectories.
cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully


## 5. Compute Protein Proxies

### 5.1 Declare Trainer

In [17]:
trainer = CpGPTTrainer() # "16-mixed" is the default precision for CpGPT inference

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/lucascamillo/pyaging/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


### 5.2 Get Predictions

In [18]:
# Get the target sample embeddings
forward_pass = trainer.predict(
    model=model,
    datamodule=datamodule,
    predict_mode="forward",
    return_keys=["pred_conditions"]
)

pred_conditions_df = pd.DataFrame(forward_pass['pred_conditions'], index=df.index, columns=model_output_features)
pred_conditions_df.head()

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


cpgpt -CpGPTDataset: Initializing class CpGPTDataset.
cpgpt -CpGPTDataset: Loaded existing dataset metrics.


/Users/lucascamillo/pyaging/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Output()

/Users/lucascamillo/pyaging/.venv/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,cpgpt_tnfsf13,cpgpt_il33,cpgpt_calca,cpgpt_npy,cpgpt_hla-dra,cpgpt_c1qa,cpgpt_fth1,cpgpt_s100b,cpgpt_ceacam5,cpgpt_mme,...,cpgpt_ccl15,cpgpt_ccl14,cpgpt_ccl13,cpgpt_ccl11,cpgpt_ccl1,cpgpt_saa1,cpgpt_s100a9,cpgpt_s100a12,cpgpt_bdnf,cpgpt_vgf
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM7625568,-0.747559,-0.964355,-0.786621,-0.229492,-0.631348,-0.725586,-0.316162,-0.497559,-0.682617,-0.341064,...,-0.830078,-0.828125,-0.845703,-0.905762,-0.754883,-0.716797,-0.326416,-0.697754,-0.329590,-0.458496
GSM7625569,-0.750977,-0.976074,-0.802246,-0.214844,-0.647949,-0.734863,-0.339600,-0.455322,-0.672363,-0.358887,...,-0.847656,-0.830078,-0.899902,-0.965332,-0.762207,-0.727051,-0.287842,-0.657715,-0.302734,-0.434814
GSM7625570,-0.700195,-0.907715,-0.733398,-0.205688,-0.605957,-0.685059,-0.285400,-0.482422,-0.632812,-0.314453,...,-0.787598,-0.752930,-0.798340,-0.857910,-0.702148,-0.685059,-0.306885,-0.636230,-0.296387,-0.437744
GSM7625571,-0.630859,-0.836426,-0.639648,-0.200195,-0.579590,-0.624512,-0.227783,-0.516602,-0.593750,-0.266357,...,-0.724609,-0.677246,-0.685547,-0.769531,-0.635742,-0.633789,-0.312500,-0.600586,-0.290039,-0.419678
GSM7625572,-0.819336,-1.074219,-0.842773,-0.267578,-0.782715,-0.812012,-0.372314,-0.520508,-0.752441,-0.416260,...,-0.920410,-0.964844,-1.028320,-1.166992,-0.854004,-0.818359,-0.221191,-0.690430,-0.362793,-0.423828


## 6. Calculate CpGPTGrimAge3

In the last step, we need to join together all features that are necessary to calculate CpGPTGrimAge3, namely:
- age: chronological age of the sample;
- GrimAge2 proxies: protein and lifestyle proxies from GrimAge version 2; 
- CpGPT protein proxies: protein levels predicted with CpGPT.

### Join All Features

In [24]:
# Get age from the metadata
age = metadata.loc[:, ['age:ch1']].astype(float)
age.columns = ['age']

# Add age to the filtered betas
df_filtered['age'] = metadata.loc[:, ['age:ch1']].astype(float)

# Get GrimAge2 proxies
grimage2_proxies = [
    "grimage2timp1",
    "grimage2packyrs",
    "grimage2logcrp",
    "grimage2b2m",
    "grimage2adm",
    "grimage2leptin",
    "grimage2gdf15",
    "grimage2pai1",
]
adata_grimage2 = pya.pp.df_to_adata(df_filtered, verbose=False)
pya.pred.predict_age(adata_grimage2, clock_names=grimage2_proxies, verbose=False)

# Combine all features
combined_df = pd.concat([age, adata_grimage2.obs, pred_conditions_df], axis=1)

combined_df.head()

/Users/lucascamillo/pyaging/.venv/lib/python3.13/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,age,grimage2timp1,grimage2packyrs,grimage2logcrp,grimage2b2m,grimage2adm,grimage2leptin,grimage2gdf15,grimage2pai1,cpgpt_tnfsf13,...,cpgpt_ccl15,cpgpt_ccl14,cpgpt_ccl13,cpgpt_ccl11,cpgpt_ccl1,cpgpt_saa1,cpgpt_s100a9,cpgpt_s100a12,cpgpt_bdnf,cpgpt_vgf
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM7625568,27.713889,28681.854845,-0.413095,-0.458249,1.259399e+06,274.198340,5669.402748,181.552676,20243.011946,-0.747559,...,-0.830078,-0.828125,-0.845703,-0.905762,-0.754883,-0.716797,-0.326416,-0.697754,-0.329590,-0.458496
GSM7625569,27.713889,28837.798239,-5.994888,-0.102234,1.283473e+06,283.409136,4974.201126,206.709373,22860.987565,-0.750977,...,-0.847656,-0.830078,-0.899902,-0.965332,-0.762207,-0.727051,-0.287842,-0.657715,-0.302734,-0.434814
GSM7625570,27.713889,29045.976269,-1.626265,-0.368756,1.284377e+06,277.655738,5838.281419,201.743571,23116.175618,-0.700195,...,-0.787598,-0.752930,-0.798340,-0.857910,-0.702148,-0.685059,-0.306885,-0.636230,-0.296387,-0.437744
GSM7625571,27.713889,29124.713192,-1.817934,-0.111815,1.285913e+06,282.345213,6018.160874,187.567168,23743.345339,-0.630859,...,-0.724609,-0.677246,-0.685547,-0.769531,-0.635742,-0.633789,-0.312500,-0.600586,-0.290039,-0.419678
GSM7625572,23.183333,27853.224618,-1.255798,-0.813700,1.146793e+06,278.339783,4103.492171,136.301906,19245.461789,-0.819336,...,-0.920410,-0.964844,-1.028320,-1.166992,-0.854004,-0.818359,-0.221191,-0.690430,-0.362793,-0.423828


### 6.2 Calculate CpGPTGrimAge3

In [25]:
adata = pya.pp.df_to_adata(combined_df, verbose=False)
pya.pred.predict_age(adata, clock_names=["cpgptgrimage3"], verbose=True)

adata.obs.head()

,cpgptgrimage3
GSM_ID,
GSM7625568,26.047242
GSM7625569,26.486934
GSM7625570,25.845736
GSM7625571,28.353959
GSM7625572,21.537933
